# Drug repurposing mapping

This notebook maps CTRP compound names/IDs to PRISM repurposing metadata and then fills missing clinical phase information using `drug_status`.

In [45]:
import pandas as pd
import numpy as np


In [46]:
CTRP = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_compound.txt",
    sep="\t",
)
CTRP.head()

,master_cpd_id,cpd_name,broad_cpd_id,top_test_conc_umol,cpd_status,inclusion_rationale,gene_symbol_of_protein_target,target_or_activity_of_compound,source_name,source_catalog_id,cpd_smiles
0,1788,CIL55,BRD-K46556387,10.0,probe,pilot-set,NaN,screening hit,Columbia University,NaN,CN(C)CCNC(=O)c1cc2CSc3cc(Cl)ccc3-c2s1
1,3588,BRD4132,BRD-K86574132,160.0,probe,chromatin;pilot-set,NaN,screening hit,ChemDiv Inc.,4998-1380,CC(C)N1C(=O)S\C(=C\c2ccc(Sc3nc4ccccc4[nH]3)o2)...
2,12877,BRD6340,BRD-K35716340,33.0,probe,chromatin;pilot-set,NaN,screening hit,ChemDiv Inc.,1988-0090,C(Cn1c2ccccc2c2ccccc12)c1nc2ccccc2[nH]1
3,17712,ML006,BRD-K89692698,530.0,probe,pilot-set,S1PR3,agonist of sphingosine 1-phosphate receptor 3,Enamine Ltd.,Z1037336336,C1CN(CCO1)c1nnc(-c2ccccc2)c(n1)-c1ccccc1
4,18311,Bax channel blocker,BRD-A18763547,33.0,probe,pilot-set,BAX,inhibitor of BAX-mediated mitochondrial cytoch...,Maybridge,RJC01737,OC(CN1CCNCC1)Cn1c2ccc(Br)cc2c2cc(Br)ccc12


In [47]:
drug_status = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/repo-drug-annotation-20200324.txt",
    sep="\t",
    skiprows=9,
)


In [48]:
prism_repurposed = pd.read_csv(
    "/Users/selin/Desktop/OncoTox/metadata/PRISM_REPURPOSED/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv"
)


In [49]:
# ---------------------------------------------------------
# STEP 1: PREP THE STRINGS (LOWERCASE EVERYTHING)
# ---------------------------------------------------------
# Force all names to lowercase so 'Paclitaxel' matches 'paclitaxel'
CTRP["cpd_name_lower"] = CTRP["cpd_name"].astype(str).str.lower()
drug_status["pert_iname_lower"] = drug_status["pert_iname"].astype(str).str.lower()
prism_repurposed["drug_name_lower"] = prism_repurposed["Drug.Name"].astype(str).str.lower()

# ---------------------------------------------------------
# STEP 2: CLEAN THE BROAD IDs IN PRISM
# ---------------------------------------------------------
# PRISM IDs look like "BRD:BRD-K00104122-001-01-9"
# CTRP IDs look like "BRD-K46556387"
# We need to extract just the "BRD-KXXXXXXX" part from PRISM to match CTRP
prism_repurposed["broad_cpd_id_clean"] = prism_repurposed["IDs"].str.extract(r"(BRD-[A-Z0-9]+)")

# ---------------------------------------------------------
# STEP 3: THE MERGES
# ---------------------------------------------------------

# PATH A: The Direct Merge (CTRP Name -> Hub Name)
direct_merged = CTRP.merge(
    drug_status,
    left_on="cpd_name_lower",
    right_on="pert_iname_lower",
    how="left",
)

# PATH B: The Bridge Merge (CTRP ID -> PRISM ID -> PRISM Name -> Hub Name)
# 1. Link CTRP to PRISM using the cleaned Broad ID
bridge_step1 = CTRP.merge(
    prism_repurposed[["broad_cpd_id_clean", "drug_name_lower", "Synonyms"]],
    left_on="broad_cpd_id",
    right_on="broad_cpd_id_clean",
    how="left",
)

# 2. Link that result to the Hub using the PRISM drug name
bridge_merged = bridge_step1.merge(
    drug_status,
    left_on="drug_name_lower",
    right_on="pert_iname_lower",
    how="left",
)

# ---------------------------------------------------------
# STEP 4: COMBINE THE RESULTS
# ---------------------------------------------------------
# Where Path A (direct merge) failed to find a clinical_phase,
# fill in the blank with the result from Path B (bridge merge).
direct_merged["clinical_phase"] = direct_merged["clinical_phase"].fillna(bridge_merged["clinical_phase"])

# You can do the same for any other columns you want to keep from drug_status
direct_merged["pert_iname"] = direct_merged["pert_iname"].fillna(bridge_merged["pert_iname"])
direct_merged["moa"] = direct_merged["moa"].fillna(bridge_merged["moa"])
direct_merged["disease_area"] = direct_merged["disease_area"].fillna(bridge_merged["disease_area"])

# ---------------------------------------------------------
# STEP 5: CLEAN UP
# ---------------------------------------------------------
# Drop the temporary lowercase matching columns to keep your dataframe tidy
test_df_final = direct_merged.drop(columns=["cpd_name_lower", "pert_iname_lower"])
test_df_final = test_df_final[[
    "master_cpd_id",
    "cpd_name",
    "broad_cpd_id",
    "cpd_status",
    "clinical_phase",
    "pert_iname",
    "moa",
]]

test_df_final.head()


,master_cpd_id,cpd_name,broad_cpd_id,cpd_status,clinical_phase,pert_iname,moa
0,1788,CIL55,BRD-K46556387,probe,NaN,NaN,NaN
1,3588,BRD4132,BRD-K86574132,probe,NaN,NaN,NaN
2,12877,BRD6340,BRD-K35716340,probe,NaN,NaN,NaN
3,17712,ML006,BRD-K89692698,probe,NaN,NaN,NaN
4,18311,Bax channel blocker,BRD-A18763547,probe,Preclinical,BAX-channel-blocker,cytochrome C release inhibitor


In [50]:
test_df_final["clinical_phase"].value_counts(dropna=False)


clinical_phase
NaN                198
Preclinical        129
Launched           106
Phase 2             38
Phase 3             32
Phase 1             30
Phase 1/Phase 2      8
Phase 2/Phase 3      4
Withdrawn            1
Name: count, dtype: int64

In [51]:
test_df_final[test_df_final["clinical_phase"].isna()]["cpd_status"].value_counts()


cpd_status
probe        159
GE-active     24
clinical      12
FDA            3
Name: count, dtype: int64

In [52]:
gdsc = pd.read_excel("../data/GDSC2_fitted_dose_response_27Oct23.xlsx")
gdsc.head()

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
0,GDSC2,343,15946310,PFSK-1,SIDM01132,Other Solid Cancers,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-1.463887,0.930220,0.089052,0.433123
1,GDSC2,343,15946548,A673,SIDM00848,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-4.869455,0.614970,0.111351,-1.421100
2,GDSC2,343,15946830,ES5,SIDM00263,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.360586,0.791072,0.142855,-0.599569
3,GDSC2,343,15947087,ES7,SIDM00269,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-5.044940,0.592660,0.135539,-1.516647
4,GDSC2,343,15947369,EW-11,SIDM00203,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.741991,0.734047,0.128059,-0.807232


In [53]:
# print where gdsc["CELL_LINE_NAME"] contains "CL34"
gdsc[gdsc["CELL_LINE_NAME"].str.contains("C32")]

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
253,GDSC2,343,16010969,C32,SIDM00890,Melanoma,1003,Camptothecin,TOP1,DNA replication,0.000100,0.1000,-2.968478,0.808096,0.105556,-0.386078
1164,GDSC2,343,16010970,C32,SIDM00890,Melanoma,1004,Vinblastine,Microtubule destabiliser,Mitosis,0.000100,0.1000,-1.761811,0.896029,0.147540,0.794979
1913,GDSC2,343,16010971,C32,SIDM00890,Melanoma,1005,Cisplatin,DNA crosslinker,DNA replication,0.004002,6.0000,2.185151,0.893228,0.084534,-0.563130
2666,GDSC2,343,16010972,C32,SIDM00890,Melanoma,1006,Cytarabine,Antimetabolite,Other,0.002001,2.0000,-1.495771,0.651483,0.132113,-1.340826
3465,GDSC2,343,16010973,C32,SIDM00890,Melanoma,1007,Docetaxel,Microtubule stabiliser,Mitosis,0.000013,0.0125,-2.830356,0.942783,0.075231,0.974381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238564,GDSC2,343,16011260,C32,SIDM00890,Melanoma,2362,THR-103,Mutant RAS,PI3K/MTOR signaling,0.010005,10.0000,5.067956,0.973952,0.083010,0.550506
239297,GDSC2,343,16011261,C32,SIDM00890,Melanoma,2438,ascorbate (vitamin C),anti-oxidant proteins,Other,2.001054,2000.0000,11.982729,0.991428,0.045740,1.352725
240029,GDSC2,343,16011262,C32,SIDM00890,Melanoma,2439,glutathione,anti-oxidant proteins,Other,0.640337,640.0000,9.388740,0.976639,0.086367,0.250423
240761,GDSC2,343,16011263,C32,SIDM00890,Melanoma,2498,alpha-lipoic acid,Metabolism,Metabolism,0.121064,121.0000,8.615496,0.987319,0.065347,0.999789


In [54]:
sc_expr_data = pd.read_csv("/Users/selin/Desktop/OncoTox/scRNAseq_SCP542/metadata/Metadata.txt", sep="\t")
sc_expr_data.head()

/var/folders/6k/gr_1_h_97154rq71pm_q3jn40000gn/T/ipykernel_18270/333420681.py:1: DtypeWarning: Columns (0: Genes_expressed, 1: SkinPig_score, 2: EMTI_score, 3: EMTII_score, 4: EMTIII_score, 5: IFNResp_score, 6: p53Sen_score, 7: EpiSen_score, 8: StressResp_score, 9: ProtMatu_score, 10: ProtDegra_score, 11: G1/S_score, 12: G2/M_score) have mixed types. Specify dtype option on import or set low_memory=False.
  sc_expr_data = pd.read_csv("/Users/selin/Desktop/OncoTox/scRNAseq_SCP542/metadata/Metadata.txt", sep="\t")


,NAME,Cell_line,Pool_ID,Cancer_type,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score
0,TYPE,group,group,group,numeric,group,group,group,group,numeric,...,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric
1,AAACCTGAGACATAAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4318,NaN,NaN,NaN,NaN,0.166,...,-0.935,-0.935,0.13,0.619,1.869,-0.004,0.805,0.896,0.424,-1.125
2,AACGTTGTCACCCGAG-1-18,NCIH2126_LUNG,18,Lung Cancer,5200,NaN,NaN,NaN,NaN,-0.213,...,-1.027,-1.027,0.066,1.049,1.267,0.252,1.299,1.61,0.624,-0.048
3,AACTGGTAGACACGAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4004,NaN,NaN,NaN,NaN,-0.101,...,-0.677,-0.677,0.304,0.822,2.401,0.141,0.451,1.225,-0.795,0.064
4,AACTGGTAGGGCTTGA-1-18,NCIH2126_LUNG,18,Lung Cancer,4295,NaN,NaN,NaN,NaN,-0.014,...,-0.735,-0.735,0.094,0.834,2.282,0.15,0.267,0.892,-0.238,1.118


In [55]:
# Find which rows in sc_expr_data have no matching cell line name with gdsc["CELL_LINE_NAME"]
sc_expr_data[~sc_expr_data["Cell_line"].isin(gdsc["CELL_LINE_NAME"].unique())]


,NAME,Cell_line,Pool_ID,Cancer_type,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score
0,TYPE,group,group,group,numeric,group,group,group,group,numeric,...,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric
1,AAACCTGAGACATAAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4318,NaN,NaN,NaN,NaN,0.166,...,-0.935,-0.935,0.13,0.619,1.869,-0.004,0.805,0.896,0.424,-1.125
2,AACGTTGTCACCCGAG-1-18,NCIH2126_LUNG,18,Lung Cancer,5200,NaN,NaN,NaN,NaN,-0.213,...,-1.027,-1.027,0.066,1.049,1.267,0.252,1.299,1.61,0.624,-0.048
3,AACTGGTAGACACGAC-1-18,NCIH2126_LUNG,18,Lung Cancer,4004,NaN,NaN,NaN,NaN,-0.101,...,-0.677,-0.677,0.304,0.822,2.401,0.141,0.451,1.225,-0.795,0.064
4,AACTGGTAGGGCTTGA-1-18,NCIH2126_LUNG,18,Lung Cancer,4295,NaN,NaN,NaN,NaN,-0.014,...,-0.735,-0.735,0.094,0.834,2.282,0.15,0.267,0.892,-0.238,1.118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53509,c4722,JHU006_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,3343,NaN,NaN,NaN,NaN,0.018,...,-0.505,-0.505,1.657,1.583,3.85,0.539,0.473,0.544,-1.079,-1.349
53510,c4724,JHU006_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,6977,NaN,NaN,NaN,NaN,-0.098,...,-0.876,-0.876,0.669,1.086,3.046,0.799,0.49,1.319,-0.37,0.057
53511,c4731,JHU006_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,6638,NaN,NaN,NaN,NaN,-0.112,...,-0.112,-0.112,0.61,0.693,2.289,0.65,0.729,1.143,-0.508,0.501
53512,c4735,JHU006_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,4052,NaN,NaN,NaN,NaN,-0.244,...,1.981,1.981,0.523,-0.309,0.267,0.822,1.049,0.777,0.296,-0.936


In [ ]:
import pandas as pd
from IPython.display import display


def norm_key(s: pd.Series) -> pd.Series:
    """Normalize cell-line names to a comparable base key."""
    x = (
        s.astype("string")
        .str.strip()
        .str.upper()
        .str.replace("-", "", regex=False)
        .str.replace(r"\s+", "", regex=True)
    )
    # step 2: drop suffix after '_' (e.g. NCIH2126_LUNG -> NCIH2126)
    return x.str.split("_", n=1).str[0]


gdsc = gdsc.copy()
sc_expr_data = sc_expr_data.copy()

gdsc["_key"] = norm_key(gdsc["CELL_LINE_NAME"])
sc_expr_data["_key"] = norm_key(sc_expr_data["Cell_line"])

# which sc rows have NO matching gdsc annotation at the cell-line key level
gdsc_keys = set(gdsc["_key"].dropna().unique())
sc_missing_mask = ~sc_expr_data["_key"].isin(gdsc_keys)
missing_rows = sc_expr_data.loc[sc_missing_mask].copy()

# 1) unmatched unique values
missing_cell_lines = (
    missing_rows[["Cell_line", "_key"]]
    .drop_duplicates()
    .sort_values(["_key", "Cell_line"])
)

print(f"sc_expr_data rows without gdsc annotation: {len(missing_rows)} / {len(sc_expr_data)}")
display(missing_cell_lines)

# 2) all rows belonging to those unmatched Cell_line values
sc_missing_rows = missing_rows.sort_values(["_key", "Cell_line"])
sc_missing_rows


sc_expr_data rows without gdsc annotation: 16643 / 53514


,Cell_line,_key
52359,93VU_UPPER_AERODIGESTIVE_TRACT,93VU
17102,ACCMESO1_PLEURA,ACCMESO1
44730,BICR16_UPPER_AERODIGESTIVE_TRACT,BICR16
21593,BICR56_UPPER_AERODIGESTIVE_TRACT,BICR56
6121,BICR6_UPPER_AERODIGESTIVE_TRACT,BICR6
...,...,...
6758,UMUC1_URINARY_TRACT,UMUC1
13783,VMCUB1_URINARY_TRACT,VMCUB1
2623,WM88_SKIN,WM88
10178,YD38_UPPER_AERODIGESTIVE_TRACT,YD38


,NAME,Cell_line,Pool_ID,Cancer_type,Genes_expressed,Discrete_cluster_minpts5_eps1.8,Discrete_cluster_minpts5_eps1.5,Discrete_cluster_minpts5_eps1.2,CNA_subclone,SkinPig_score,...,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score,_key
52359,c12,93VU_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,5171,NaN,NaN,NaN,NaN,-0.238,...,-0.747,-0.323,-0.039,1.002,-0.504,0.142,0.429,-0.703,1.23,93VU
52360,c27,93VU_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,2491,NaN,NaN,NaN,NaN,0.251,...,0.049,0.486,1.229,2.028,0.598,-0.196,0.777,-0.206,-0.369,93VU
52361,c38,93VU_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,3474,NaN,NaN,NaN,NaN,0.209,...,-0.282,0.127,0.014,1.323,-0.38,0.562,0.522,0.73,0.392,93VU
52362,c49,93VU_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,2469,NaN,NaN,NaN,NaN,-0.003,...,-0.671,0.051,0.137,0.722,-0.126,0.364,0.837,0.443,0.941,93VU
52363,c52,93VU_UPPER_AERODIGESTIVE_TRACT,custom,Head and Neck Cancer,3488,NaN,NaN,NaN,NaN,-0.101,...,-0.669,-0.127,-0.18,0.867,0.285,0.516,1.421,-0.399,-0.294,93VU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35137,TTGCCGTAGCCAGGAT-12-10,ZR751_BREAST,10,Breast Cancer,3717,NaN,NaN,NaN,NaN,-0.19,...,-1.583,-0.424,0.351,0.82,-0.313,-0.733,-0.541,0.551,-0.056,ZR751
35138,TTGCCGTTCTTCATGT-12-10,ZR751_BREAST,10,Breast Cancer,2278,NaN,NaN,NaN,NaN,0.045,...,-0.948,-0.365,0.093,0.673,0.02,-0.594,-1.429,1.053,0.006,ZR751
35139,TTTATGCTCGGATGGA-12-10,ZR751_BREAST,10,Breast Cancer,3045,NaN,NaN,NaN,NaN,-0.172,...,-1.11,-0.098,0.056,0.708,0.123,-0.457,-1.086,0.306,0.234,ZR751
35140,TTTGCGCAGCCACTAT-12-10,ZR751_BREAST,10,Breast Cancer,3726,NaN,NaN,NaN,NaN,-0.161,...,-1.279,-0.494,1.203,1.072,-0.309,-0.625,-1.13,-0.2,-1.14,ZR751


In [57]:
merged.head()

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,...,EMTII_score,EMTIII_score,IFNResp_score,p53Sen_score,EpiSen_score,StressResp_score,ProtMatu_score,ProtDegra_score,G1/S_score,G2/M_score
0,GDSC2,343,15947651,SK-ES-1,SIDM01111,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,...,-0.7,-0.7,-0.852,-1.417,-1.095,-0.465,0.198,0.496,0.242,1.0
1,GDSC2,343,15947651,SK-ES-1,SIDM01111,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,...,-0.578,-0.578,-0.438,-1.354,-0.997,-0.095,0.364,0.599,0.104,-0.401
2,GDSC2,343,15947651,SK-ES-1,SIDM01111,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,...,-0.308,-0.308,-0.676,-1.358,-1.157,-0.389,-0.238,0.037,-0.072,1.25
3,GDSC2,343,15947651,SK-ES-1,SIDM01111,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,...,-0.484,-0.484,-0.543,-1.26,-0.935,-0.378,0.469,0.497,-0.591,-0.006
4,GDSC2,343,15947651,SK-ES-1,SIDM01111,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,...,-0.474,-0.474,-0.331,0.042,-0.205,0.302,-0.026,-0.172,-0.641,-0.674
